In [0]:
# ===================================================
# BLOCK 1 — PARAMETERS AND CONTRACTS (PYTHON)
# ===================================================

"""
Define the governed tables and source locations that must be available before
the workflow executes data-quality reconciliation.
"""

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("job_run_id", "MANUAL", "Lakeflow Job Run ID")

JOB_RUN_ID = dbutils.widgets.get("job_run_id")

ARRIVAL_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/"
    "external_source/streaming_demo/input"
)

REQUIRED_TABLES = [
    "semiconplus_portfolio.bronze.production_lots",
    "semiconplus_portfolio.bronze.streaming_test_results",
    "semiconplus_portfolio.silver.streaming_test_results",
    "semiconplus_portfolio.silver.streaming_late_test_results",
    "semiconplus_portfolio.quarantine.streaming_test_results",
    "semiconplus_portfolio.gold.mart_streaming_yield_5m",
]

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY (PYTHON)
# ===================================================

"""
Fail before expensive reconciliation queries when a required governed table is
missing or was not published successfully by an upstream pipeline.
"""

table_results = []

for table_name in REQUIRED_TABLES:
    table_exists = spark.catalog.tableExists(table_name)
    table_results.append((table_name, table_exists))

display(
    spark.createDataFrame(
        table_results,
        ["table_name", "table_exists"],
    )
)

missing_tables = [
    table_name
    for table_name, table_exists in table_results
    if not table_exists
]

assert not missing_tables, (
    f"Required governed tables are missing: {missing_tables}"
)

In [0]:
# ===================================================
# BLOCK 3 — SOURCE-DIRECTORY AVAILABILITY (PYTHON)
# ===================================================

"""
Validate that the controlled streaming landing directory is readable and
contains the ten files required by the completed two-wave demonstration.
"""

arrival_files = sorted(
    item.name
    for item in dbutils.fs.ls(ARRIVAL_DIRECTORY)
    if not item.isDir() and item.name.lower().endswith(".json")
)

print(f"Controlled arrival files: {len(arrival_files)}")
print("\n".join(arrival_files))

assert len(arrival_files) == 10, (
    f"Expected 10 controlled arrival files, found {len(arrival_files)}."
)

In [0]:
# ===================================================
# BLOCK 4 — MINIMUM DATA AVAILABILITY (PYTHON)
# ===================================================

"""
Reject empty data products before downstream validation so a technically
successful but data-empty pipeline cannot be reported as operationally healthy.
"""

minimum_results = []

for table_name in REQUIRED_TABLES:
    row_count = spark.table(table_name).limit(1).count()
    contains_data = row_count > 0

    minimum_results.append(
        (table_name, contains_data)
    )

display(
    spark.createDataFrame(
        minimum_results,
        ["table_name", "contains_data"],
    )
)

empty_tables = [
    table_name
    for table_name, contains_data in minimum_results
    if not contains_data
]

# Quarantine is allowed to be empty when every source record meets the schema
# and business contract.
allowed_empty_tables = {
    "semiconplus_portfolio.quarantine.streaming_test_results"
}

unexpected_empty_tables = [
    table_name
    for table_name in empty_tables
    if table_name not in allowed_empty_tables
]

assert not unexpected_empty_tables, (
    f"Required data products are empty: {unexpected_empty_tables}"
)

print("PREFLIGHT CHECKS: PASSED")